# unlocr on GPU (Unlimited-OCR GGUF via llama.cpp + CUDA)

OCR PDFs with the `unlocr` CLI driving a **GPU-accelerated `llama-server`** that serves the `baidu/Unlimited-OCR` model (DeepSeek-OCR architecture) in GGUF form.

**Why this path (not vLLM):** the official Unlimited-OCR vLLM build ships only as a Docker image, and a stock Colab runtime has no Docker daemon. llama.cpp runs from a normal binary, fits in less VRAM, and `unlocr` already knows how to drive it (downloads the GGUF, sets `--chat-template deepseek-ocr` + `--mmproj`). We just build a CUDA `llama-server` and tell it to offload to the GPU.

How it fits together:
- The Unlimited-OCR architecture needs a **PR-branch build of llama.cpp** (PR #24975); stock `llama-server` will not load the weights.
- `unlocr` spawns `llama-server`; `server_args` does not pass `-ngl`, so we set **`LLAMA_ARG_N_GPU_LAYERS=99`** in the environment and `llama-server` offloads every layer to the GPU. No CLI flags, no source changes.
- The GGUF weights (`--quality good` = Q8_0, ~2.9 GB) download from Hugging Face on first run and are cached.

**Before you start:** **Runtime > Change runtime type > GPU (T4 / L4 / A100)**.

Cells: confirm GPU -> install poppler + build `llama-server` (CUDA) -> fetch `unlocr` CLI -> enable GPU offload -> **upload one PDF or a batch** -> OCR -> download results.

For CPU/quantized use off Colab, see the [unlocr README](https://github.com/whit3rabbit/unlocr).

In [ ]:
# 0. Confirm a GPU is attached (fails loudly if you forgot to pick a GPU runtime).
!nvidia-smi || (echo 'No GPU. Runtime > Change runtime type > GPU (T4/L4/A100).' && false)

In [ ]:
# 1. System deps: poppler-utils (pdftoppm, used by unlocr to rasterize PDFs) and
#    the toolchain to build llama.cpp (cmake + compiler). CUDA itself is already
#    present on a GPU runtime.
!apt-get -qq update && apt-get -qq install -y poppler-utils build-essential cmake git
!pdftoppm -v

In [ ]:
# 2. Build llama-server with CUDA from the Unlimited-OCR (DeepSeek-OCR) support
#    branch (PR #24975). Stock llama.cpp cannot load these weights.
#    This compiles for a few minutes. -DLLAMA_CURL=OFF drops the libcurl dep
#    (unlocr passes a local model path, so the server never fetches over HTTP).
%%bash
set -e
cd /content
if [ ! -x /usr/local/bin/llama-server ]; then
  rm -rf llama.cpp
  git clone --quiet https://github.com/ggml-org/llama.cpp
  cd llama.cpp
  git fetch --quiet origin pull/24975/head:pr24975
  git checkout --quiet pr24975
  cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON -DLLAMA_CURL=OFF
  cmake --build build -j --target llama-server
  install -m 0755 build/bin/llama-server /usr/local/bin/llama-server
fi
llama-server --version 2>&1 | head -n 3

In [ ]:
# 3. Get the unlocr Linux x86_64 binary from a GitHub release.
#    Tries the 'latest' release, then falls back to a pinned tag (the first
#    release). If neither has a matching asset, run the build-from-source cell.
import json, os, tarfile, urllib.request

UNLOCR = "/content/unlocr"
REPO = "whit3rabbit/unlocr"
PINNED_TAG = "v0.1.0"  # bump as releases advance; used when 'latest' is unavailable

def fetch_release(url):
    try:
        return json.load(urllib.request.urlopen(url))
    except Exception as e:
        print("  release lookup failed:", url, "->", e)
        return None

def pick_asset(rel):
    if not rel:
        return None
    return next(
        (a for a in rel.get("assets", [])
         if ("linux" in a["name"].lower())
         and ("x86_64" in a["name"].lower() or "x64" in a["name"].lower() or "amd64" in a["name"].lower())
         and a["name"].endswith((".tar.gz", ".tgz"))),
        None,
    )

rel = fetch_release(f"https://api.github.com/repos/{REPO}/releases/latest")
asset = pick_asset(rel)
if asset is None:
    print(f"No usable 'latest' asset; trying pinned tag {PINNED_TAG}.")
    rel = fetch_release(f"https://api.github.com/repos/{REPO}/releases/tags/{PINNED_TAG}")
    asset = pick_asset(rel)
if asset is None:
    raise SystemExit(
        "No Linux x86_64 release tarball found for unlocr. "
        "Run the 'Build from source' cell below instead."
    )

print("Downloading", asset["name"])
urllib.request.urlretrieve(asset["browser_download_url"], "/content/unlocr.tar.gz")
with tarfile.open("/content/unlocr.tar.gz") as t:
    member = next(m for m in t.getmembers() if os.path.basename(m.name) == "unlocr")
    member.name = os.path.basename(member.name)
    t.extract(member, "/content")
os.chmod(UNLOCR, 0o755)
!{UNLOCR} --version

In [ ]:
# 3b. Build from source (fallback). Only run this if the release-download cell failed.
#     Compiles the CLI with cargo; takes a few minutes.
# !curl -sSf https://sh.rustup.rs | sh -s -- -y
# !. "$HOME/.cargo/env" && cargo install --git https://github.com/whit3rabbit/unlocr unlocr --root /content/.cargo
# import os; UNLOCR = "/content/.cargo/bin/unlocr"
# !{UNLOCR} --version

In [ ]:
# 4. Enable GPU offload. unlocr spawns llama-server itself (once per run) and does
#    not pass -ngl, so we set it via the env var llama-server reads at startup.
#    99 = offload all layers. This os.environ entry is inherited by the `!unlocr`
#    cells below. No server is started here; unlocr starts and stops it per run.
import os, shutil

os.environ["LLAMA_ARG_N_GPU_LAYERS"] = "99"

assert shutil.which("llama-server"), "llama-server not on PATH; re-run the build cell."
print("llama-server:", shutil.which("llama-server"))
print("GPU offload set: LLAMA_ARG_N_GPU_LAYERS=", os.environ["LLAMA_ARG_N_GPU_LAYERS"])

In [ ]:
# 5. Upload a PDF to OCR.
from google.colab import files
uploaded = files.upload()
PDF = next(iter(uploaded))
print("Uploaded:", PDF)

In [ ]:
# 6. OCR the single uploaded PDF. unlocr downloads the Unlimited-OCR GGUF on first
#    run (--quality good = Q8_0, ~2.9 GB; cached afterwards), spawns the CUDA
#    llama-server, and OCRs every page. Use --quality less (Q4_K_M) if VRAM is tight.
!{UNLOCR} "{PDF}" --quality good -o /content/out.md
print(open("/content/out.md").read()[:2000])

In [ ]:
# 7. Download the markdown result.
from google.colab import files
files.download("/content/out.md")

## Bulk: OCR many PDFs at once

The cells above handle one PDF. For a batch, upload several PDFs into a folder and let `unlocr` process the whole folder (it accepts files, folders, and globs as positional inputs). Each `foo.pdf` becomes `foo.md` under the output directory.

Steps 2 and 4 (the CUDA `llama-server` build and `LLAMA_ARG_N_GPU_LAYERS`) must have run; `unlocr` starts and stops the server itself per batch.

In [ ]:
# 8. Upload many PDFs into /content/pdfs (re-run to add more).
import os, shutil
from google.colab import files

IN_DIR = "/content/pdfs"
OUT_DIR = "/content/out"
os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

uploaded = files.upload()  # pick several PDFs in the dialog
for name in uploaded:
    shutil.move(name, os.path.join(IN_DIR, os.path.basename(name)))

pdfs = sorted(f for f in os.listdir(IN_DIR) if f.lower().endswith(".pdf"))
print(f"{len(pdfs)} PDF(s) queued in {IN_DIR}:")
for f in pdfs:
    print(" ", f)

In [ ]:
# 9. OCR the whole folder. unlocr loops every PDF; foo.pdf -> OUT_DIR/foo.md.
#    The llama-server is reused across pages; --recursive also picks up subfolders.
!{UNLOCR} "{IN_DIR}" --recursive --quality good --out "{OUT_DIR}"
print("\nWrote:")
!ls -la "{OUT_DIR}"

In [ ]:
# 10. Zip all markdown results and download.
import shutil
from google.colab import files

shutil.make_archive("/content/unlocr_out", "zip", OUT_DIR)
files.download("/content/unlocr_out.zip")